## inference code for whisper small

In [1]:
from safetensors import safe_open
import json

file_location = "model.safetensors"

with safe_open(file_location, framework="pt", device='cpu') as f: 
    model_structure = {
        layer: list(f.get_tensor(layer).shape)
        for layer in f.keys()
    }

    with open("model_safetensors.json", "w") as json_file:
        json.dump(model_structure, json_file, indent=4)
    
    for layer, shape in model_structure.items():
        if 'model.decoder.layers' in layer:
            print(f"Name: {layer}, shape: {shape}")

Name: model.decoder.layers.0.encoder_attn.k_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.out_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.out_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.q_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.q_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.v_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.v_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn_layer_norm.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn_layer_norm.weight, shape: [768]
Name: model.decoder.layers.0.fc1.bias, shape: [3072]
Name: model.decoder.layers.0.fc1.weight, shape: [3072, 768]
Name: model.decoder.layers.0.fc2.bias, shape: [768]
Name: model.decoder.layers.0.fc2.weight, shape: [768, 3072]
Name: model.decoder.layers.0.final_layer_norm.bias, shape: [768]
Name: model.decoder.layers.0.final_layer_norm.weight, sh

In [2]:
import torch
from safetensors import safe_open

# In Whisper, k_proj has no bias across all three attention modules.
# (encoder self-attention, decoder self-attention, and decoder cross-attention).


class WhisperEncoderAttention(torch.nn.Module):
    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["encoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # loading pretrained weights
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.out_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x):
        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        return self.out_proj(out)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [3]:
class WhisperEncoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.d_ff = config["encoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # loading pretrained weights
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.weight")
        )
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.weight")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [4]:
class WhisperEncoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no, safetensor_files: list):
        super().__init__()

        self.files = safetensor_files

        self.encoder_attn = WhisperEncoderAttention(config, layer_no, self.files)
        self.encoder_ffn = WhisperEncoderFeedForwardNeuralNetwork(
            config, layer_no, self.files
        )

        self.attn_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )

        self.final_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.weight")
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        residual = x
        x = residual + self.encoder_attn(self.attn_layer_norm(x))

        residual = x
        x = residual + self.encoder_ffn(self.final_layer_norm(x))

        return x

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [5]:
class WhisperEncoder(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list):
        super().__init__()

        self.model_config = config
        self.files = safetensor_files

        self.conv1 = torch.nn.Conv1d(
            in_channels=self.model_config["num_mel_bins"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            padding=1,
        )
        self.conv1.weight.data.copy_(self.get_tensor("model.encoder.conv1.weight"))
        self.conv1.bias.data.copy_(self.get_tensor("model.encoder.conv1.bias"))

        self.conv2 = torch.nn.Conv1d(
            in_channels=self.model_config["d_model"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            stride=2,
            padding=1,
        )
        self.conv2.weight.data.copy_(self.get_tensor("model.encoder.conv2.weight"))
        self.conv2.bias.data.copy_(self.get_tensor("model.encoder.conv2.bias"))

        self.embed_positions = torch.nn.Embedding(
            num_embeddings=self.model_config["max_source_positions"],
            embedding_dim=self.model_config["d_model"],
        )
        self.embed_positions.weight.data.copy_(
            self.get_tensor("model.encoder.embed_positions.weight")
        )

        self.layers = torch.nn.ModuleList(
            [
                WhisperEncoderLayer(self.model_config, i, self.files)
                for i in range(self.model_config["encoder_layers"])
            ]
        )

        self.layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.layer_norm.weight.data.copy_(
            self.get_tensor("model.encoder.layer_norm.weight")
        )
        self.layer_norm.bias.data.copy_(
            self.get_tensor("model.encoder.layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        x = torch.nn.functional.gelu(self.conv1(x))
        x = torch.nn.functional.gelu(self.conv2(x))

        x = x.transpose(-2, -1)

        positions = torch.arange(x.size(1), device=x.device)

        x = x + self.embed_positions(positions)

        for layer in self.layers:
            x = layer(x)

        return self.layer_norm(x)

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

## Decoder

In [6]:
# Only One Seq will be came for process.

class WhisperDecoderMaskedAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    config["max_target_positions"], config["max_target_positions"]
                ),
                diagonal=1,
            ).bool(),
        )

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.out_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x, past_kv: None):

        B, S, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv

            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present_kv = (k, v)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)

        if S > 1:
            attn_score = attn_score.masked_fill(self.mask[:S, :S], float("-inf"))

        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        out = self.out_proj(out)

        return out, present_kv

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [7]:
class WhisperDecoderCrossAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.weight"
            )
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.k_proj.weight"
            )
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.weight"
            )
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.bias"
            )
        )

    @torch.no_grad()
    def forward(self, decoder_rep, encoder_rep, past_kv: None):

        DB, DS, _ = decoder_rep.shape
        EB, ES, _ = encoder_rep.shape

        if past_kv is not None:
            k, v = past_kv

        else:
            k = self.k_proj(encoder_rep)
            v = self.v_proj(encoder_rep)

            k = k.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)
            v = v.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)

        q = self.q_proj(decoder_rep)
        q = q.view(DB, DS, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(DB, DS, self.d_model)

        out = self.out_proj(out)

        return out, (k, v)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [8]:
class WhisperDecoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()
        self.d_model = config["d_model"]
        self.d_ff = config["decoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # weight's
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.weight")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.weight")
        )

        # bias
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [9]:
class WhisperDecoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.files = safetensor_files

        self.decoder_mask_attn = WhisperDecoderMaskedAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_cross_attn = WhisperDecoderCrossAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_ffn = WhisperDecoderFeedForwardNeuralNetwork(
            config, layer_no, safetensor_files
        )

        # layer norm
        self.mask_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.cross_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.final_layer_norm = torch.nn.LayerNorm(self.d_model)

        # weight's
        self.mask_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.cross_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.weight"
            )
        )
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.weight")
        )

        # bias
        self.mask_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )
        self.cross_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.bias"
            )
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(
        self, from_decoder, from_encoder, masked_attn_kv: None, cross_attn_kv: None
    ):

        residual = from_decoder

        x, present_masked_attn_kv = self.decoder_mask_attn(
            self.mask_attn_layer_norm(residual), masked_attn_kv
        )
        x = residual + x

        residual = x

        x, present_cross_attn_kv = self.decoder_cross_attn(
            self.cross_attn_layer_norm(x), from_encoder, cross_attn_kv
        )
        x = residual + x

        x = x + self.decoder_ffn(self.final_layer_norm(x))

        return x, present_masked_attn_kv, present_cross_attn_kv

    def get_tensor(self, name):
        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [10]:
class WhisperDecoder(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list):
        super().__init__()

        self.model_config = config
        self.files = safetensor_files

        self.embed_tokens = torch.nn.Embedding(
            num_embeddings=self.model_config["vocab_size"],
            embedding_dim=self.model_config["d_model"],
            padding_idx=self.model_config["pad_token_id"],
        )
        self.embed_positions = torch.nn.Embedding(
            num_embeddings=self.model_config["max_target_positions"],
            embedding_dim=self.model_config["d_model"],
        )
        self.layers = torch.nn.ModuleList(
            [
                WhisperDecoderLayer(self.model_config, i, self.files)
                for i in range(self.model_config["decoder_layers"])
            ]
        )
        self.layer_norm = torch.nn.LayerNorm(self.model_config["d_model"])

        # weight's
        self.embed_tokens.weight.data.copy_(
            self.get_tensor("model.decoder.embed_tokens.weight")
        )
        self.embed_positions.weight.data.copy_(
            self.get_tensor("model.decoder.embed_positions.weight")
        )
        self.layer_norm.weight.data.copy_(
            self.get_tensor("model.decoder.layer_norm.weight")
        )
        # bias
        self.layer_norm.bias.data.copy_(
            self.get_tensor("model.decoder.layer_norm.bias")
        )
        # kv cache
        self.masked_attn_kv = [None] * self.model_config["decoder_layers"]
        self.cross_attn_kv = [None] * self.model_config["decoder_layers"]

        self.current_pos = None

    @torch.no_grad()
    def forward(self, from_decoder, from_encoder):

        x = self.embed_tokens(from_decoder)

        if self.current_pos is None:
            self.current_pos = x.size(1)
            positions = torch.arange(self.current_pos, device=x.device)
            x = x + self.embed_positions(positions)

        else:
            position = torch.tensor([self.current_pos], device=x.device)
            x =  x + self.embed_positions(position)
            self.current_pos += 1

        for i, layer in enumerate(self.layers):
            x, self.masked_attn_kv[i], self.cross_attn_kv[i] = layer(
                x, from_encoder, self.masked_attn_kv[i], self.cross_attn_kv[i]
            )

        return self.layer_norm(x)

    def get_tensor(self, name):
        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

    def clean_cache(self):
        self.masked_attn_kv = [None] * self.model_config["decoder_layers"]
        self.cross_attn_kv = [None] * self.model_config["decoder_layers"]
        self.current_pos = None

In [66]:
class WhisperSmall(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list[str]):
        super().__init__()

        self.model_config = config

        self.bos_token_id = config["bos_token_id"]
        self.decoder_start_token_id = config["decoder_start_token_id"]
        self.eos_token_id = config["eos_token_id"]
        self.max_length = config["max_length"]

        # Whisper timestamp tokens
        self.timestamp_begin = 50364
        self.timestamp_rate = 0.02
        self.no_timestamps_token_id = 50363

        self.max_initial_timestamp = 1.0
        self.max_initial_timestamp_index = int(
            self.max_initial_timestamp / self.timestamp_rate
        )

        self.files = [
            safe_open(file, framework="pt", device="cpu") for file in safetensor_files
        ]

        self.encoder = WhisperEncoder(self.model_config, self.files)
        self.decoder = WhisperDecoder(self.model_config, self.files)

        # Tied output projection
        self.back_proj = torch.nn.Embedding(
            num_embeddings=self.model_config["vocab_size"],
            embedding_dim=self.model_config["d_model"],
        )

        self.back_proj.weight.data.copy_(
            self.get_tensor("model.decoder.embed_tokens.weight")
        )

    def is_timestamp_token(self, token_id):
        return token_id >= self.timestamp_begin

    def token_to_timestamp(self, token_id):
        return (token_id - self.timestamp_begin) * self.timestamp_rate

    def apply_timestamp_rules(self, logits, generated_tokens):

        logits = logits[:, -1, :]

        # Suppress <|notimestamps|>
        logits[:, self.no_timestamps_token_id] = -float("inf")

        # Timestamp rules
        if len(generated_tokens) > 0:

            last_was_timestamp = (
                len(generated_tokens) >= 1
                and generated_tokens[-1] >= self.timestamp_begin
            )

            penultimate_was_timestamp = (
                len(generated_tokens) < 2
                or generated_tokens[-2] >= self.timestamp_begin
            )

            # Timestamps have to appear in pairs
            if last_was_timestamp:

                if penultimate_was_timestamp:
                    # Two consecutive timestamps:
                    # next token must be normal text/EOT.
                    logits[:, self.timestamp_begin:] = -float("inf")

                else:
                    # One timestamp:
                    # next token must be another timestamp.
                    logits[:, :self.eos_token_id] = -float("inf")

            timestamps = torch.tensor(
                [
                    token
                    for token in generated_tokens
                    if token >= self.timestamp_begin
                ],
                device=logits.device,
            )

            if timestamps.numel() > 0:

                # Timestamps cannot decrease.
                #
                # If we just generated the first timestamp of a pair,
                # the second timestamp must be strictly greater.
                if last_was_timestamp and not penultimate_was_timestamp:

                    timestamp_last = timestamps[-1]

                else:

                    timestamp_last = timestamps[-1] + 1

                logits[
                    :,
                    self.timestamp_begin:timestamp_last
                ] = -float("inf")

        # First generated token must be a timestamp
        if len(generated_tokens) == 0:

            logits[:, :self.timestamp_begin] = -float("inf")

            # Maximum initial timestamp = 1 second
            last_allowed = (
                self.timestamp_begin
                + self.max_initial_timestamp_index
            )

            logits[:, last_allowed + 1:] = -float("inf")

        # ---------------------------------------------------------
        # If the combined probability of all timestamp tokens is
        # greater than the probability of the best text token,
        # force a timestamp.
        # ---------------------------------------------------------

        logprobs = torch.nn.functional.log_softmax(
            logits.float(),
            dim=-1,
        )

        timestamp_logprob = torch.logsumexp(
            logprobs[:, self.timestamp_begin:],
            dim=-1,
        )

        max_text_token_logprob = torch.max(
            logprobs[:, :self.timestamp_begin],
            dim=-1,
        ).values

        if timestamp_logprob[0] > max_text_token_logprob[0]:

            logits[:, :self.timestamp_begin] = -float("inf")

        return logits



    @torch.no_grad()
    def forward(self, x):

        generated_tokens = []

        encoder_output = self.encoder(x)

        # Whisper decoder starts with <|startoftranscript|>
        decoder_input = torch.tensor(
            [
                [
                    self.decoder_start_token_id,  # 50258
                    50259,  # <|en|>
                    50359,  # <|transcribe|>
                ]
            ],
            device=x.device,
        )

        self.decoder.clean_cache()

        for _ in range(self.max_length):

            decoder_output = self.decoder(decoder_input, encoder_output)

            logits = decoder_output @ self.back_proj.weight.T

            next_token_logits = self.apply_timestamp_rules(logits, generated_tokens)

            predicted_token = torch.argmax(next_token_logits, dim=-1).item()

            generated_tokens.append(predicted_token)

            if self.is_timestamp_token(predicted_token):

                timestamp = self.token_to_timestamp(predicted_token)

                print(f"Timestamp: {timestamp:.2f}")

            else:

                print(f"Token: {predicted_token}")

            if predicted_token == self.eos_token_id:

                print("Model Stopped")
                break

            decoder_input = torch.tensor(
                [[predicted_token]], device=decoder_output.device
            )

        return generated_tokens

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

        raise KeyError(f"Tensor not found: {name}")

In [67]:
import json, whisper

config_file_location = "config.json"

with open(config_file_location, "r") as f:
    config_file = json.load(f)

# device
device = torch.device("cpu")

# Input
audio = whisper.load_audio('../SttInput.mp3')
chunk = whisper.pad_or_trim(audio)
mel_sep = whisper.log_mel_spectrogram(chunk, n_mels=80)

model = WhisperSmall(config_file, ["model.safetensors"]).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {num_params} | In Md: {num_params * 2 / 1024 **2}")
model_output = model(mel_sep.unsqueeze(0))

Total number of parameters: 281567232 | In Md: 537.046875
Timestamp: 0.00
Token: 1033
Token: 11
Token: 40508
Token: 11
Token: 360
Token: 300
Token: 707
Token: 4489
Token: 797
Token: 0
Timestamp: 3.00
Timestamp: 3.00
Token: 1033
Token: 0
Timestamp: 5.00
Timestamp: 5.00
Token: 10223
Token: 13
Timestamp: 6.00
Timestamp: 6.00
Token: 876
Token: 11
Token: 561
Token: 366
Token: 799
Token: 959
Token: 300
Token: 13
Timestamp: 7.00
Timestamp: 7.00
Token: 286
Token: 727
Token: 1159
Token: 341
Token: 439
Token: 786
Token: 13
Timestamp: 9.00
Timestamp: 9.00
Token: 2102
Token: 366
Token: 291
Token: 30
Timestamp: 10.00
Timestamp: 10.00
Token: 876
Token: 11
Token: 2232
Token: 11
Token: 21488
Token: 62
Token: 3405
Token: 11901
Token: 13
Timestamp: 12.00
Timestamp: 12.00
Token: 708
Token: 366
Token: 291
Token: 884
Token: 510
Token: 30
Timestamp: 14.00
Timestamp: 14.00
Token: 509
Token: 445
Token: 1027
Token: 1071
Token: 2183
Token: 281
Token: 452
Token: 2954
Token: 707
Token: 2013
Token: 13
Timestamp: 1

In [63]:
from tokenizers import Tokenizer

tk = Tokenizer.from_file("tokenizer.json")

text = tk.decode(model_output)
print(text)

<|0.00|> Okay, sweetie, do that little dance again!<|3.00|><|3.00|> Okay!<|5.00|><|5.00|> Post.<|6.00|><|6.00|> Oh, people are gonna love that.<|7.00|><|7.00|> I could watch this all day.<|9.00|><|9.00|> Who are you?<|10.00|><|10.00|> Oh, uh, Todd_1975.<|12.00|><|12.00|> What are you doing here?<|14.00|><|14.00|> You just made another post to my favorite little girl.<|16.00|><|16.00|> She's almost six now, huh?<|18.00|><|18.00|> Um, yeah.<|19.00|><|19.00|> She's looking for you.<|20.00|>
